[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/32_topk_sampling_solution.ipynb)

# ✅ Solution: Top-k / Top-p Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution


In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge')
except ImportError:
    pass


In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import math


In [ ]:
# ✅ SOLUTION

import jax, jax.numpy as jnp
def sample_top_k_top_p(logits,top_k=0,top_p=1.0,temperature=1.0,key=None):
    z=logits/max(temperature,1e-8)
    if top_k>0: z=jnp.where(z>=jax.lax.top_k(z,min(top_k,z.size))[0][-1],z,-jnp.inf)
    if top_p<1:
        order=jnp.argsort(z)[::-1]; sz=z[order]; mask=(jnp.cumsum(jax.nn.softmax(sz))-jax.nn.softmax(sz))>top_p; z=z.at[order].set(jnp.where(mask,-jnp.inf,sz))
    return int(jax.random.categorical(jax.random.PRNGKey(0) if key is None else key,z))


In [ ]:
# Verify
print(sample_top_k_top_p)


In [ ]:
from jax_judge import check
check("topk_sampling")
